# Load Model and Create Submission

## Overview

This notebook loads pre-trained model weights and generates predictions for the test set.
It uses BOTH T5-small and mT5-small models for ensemble predictions.

Useful for:
- Loading models trained in previous sessions
- Using pre-trained models from the model hub
- Generating submissions without retraining

## Streaming Mode

This notebook processes data in a streaming fashion to handle large test files:
- Reads input one line at a time
- Generates predictions in batches
- Writes output one line at a time

In [ ]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, MT5ForConditionalGeneration, AutoTokenizer
import os
import csv
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Both Models from Disk or Hub

In [ ]:
# Load all 4 ensemble models
models = {}
tokenizers = {}

model_names = [
    't5_small_seed_1',
    't5_small_seed_2', 
    't5_small_seed_3',
    'mt5_small'
]

for model_name in model_names:
    model_path = f"./ensemble_models/{model_name}"
    print(f"\nLoading {model_name}...")
    if os.path.exists(model_path):
        try:
            if 'mt5' in model_name:
                tokenizers[model_name] = AutoTokenizer.from_pretrained(model_path)
                models[model_name] = MT5ForConditionalGeneration.from_pretrained(model_path)
            else:
                tokenizers[model_name] = T5Tokenizer.from_pretrained(model_path)
                models[model_name] = T5ForConditionalGeneration.from_pretrained(model_path)
            models[model_name] = models[model_name].to(DEVICE)
            print(f"  Loaded {model_name}")
        except Exception as e:
            print(f"  Failed to load {model_name}: {e}")
            models[model_name] = None
    else:
        print(f"  Model not found at {model_path}")

# Fallback: load base models if ensemble not found
if not models:
    print("\nNo ensemble models found. Loading base models...")
    tokenizers['t5'] = T5Tokenizer.from_pretrained("t5-small")
    models['t5'] = T5ForConditionalGeneration.from_pretrained("t5-small")
    models['t5'] = models['t5'].to(DEVICE)
    try:
        tokenizers['mt5'] = AutoTokenizer.from_pretrained("google/mt5-small")
        models['mt5'] = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
        models['mt5'] = models['mt5'].to(DEVICE)
    except:
        models['mt5'] = None

print(f"\nModels loaded: {list(models.keys())}")

## 2. Generate Predictions (Streaming Mode)

In [ ]:
BATCH_SIZE = 16

def generate_predictions_streaming(test_file, output_file, models, tokenizers, device, batch_size=16):
    """Process test data in streaming mode - reads one line at a time, writes one line at a time."""
    
    with open(test_file, 'r', encoding='utf-8') as infile, \
         open(output_file, 'w', encoding='utf-8', newline='') as outfile:
        
        reader = csv.DictReader(infile)
        fieldnames = ['id', 'translation']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        
        batch_ids = []
        batch_texts = []
        batch_rows = []
        total_processed = 0
        
        for row in tqdm(reader, desc="Processing rows"):
            batch_ids.append(row['id'])
            batch_texts.append("translate Akkadian to English: " + str(row['transliteration']))
            batch_rows.append(row)
            
            if len(batch_texts) >= batch_size:
                preds = process_batch(batch_texts, models, tokenizers, device)
                for row_id, pred in zip(batch_ids, preds):
                    writer.writerow({'id': row_id, 'translation': pred})
                total_processed += len(batch_texts)
                batch_ids = []
                batch_texts = []
                batch_rows = []
        
        if batch_texts:
            preds = process_batch(batch_texts, models, tokenizers, device)
            for row_id, pred in zip(batch_ids, preds):
                writer.writerow({'id': row_id, 'translation': pred})
            total_processed += len(batch_texts)
        
    return total_processed


def process_batch(texts, models, tokenizers, device):
    """Process a batch of texts through all models and combine predictions."""
    all_predictions = {}
    
    for model_name, model in models.items():
        if model is not None:
            tokenizer = tokenizers[model_name]
            
            inputs = tokenizer(
                texts,
                max_length=128,
                padding=True,
                truncation=True,
                return_tensors='pt'
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            outputs = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_length=128,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
            
            all_predictions[model_name] = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    final_predictions = []
    for i in range(len(texts)):
        if len(all_predictions) > 1:
            preds = [all_predictions[model][i] for model in all_predictions]
            final_predictions.append(max(preds, key=len))
        else:
            final_predictions.append(list(all_predictions.values())[0][i])
    
    return final_predictions


print("Starting streaming prediction...")
total = generate_predictions_streaming('test.csv', 'submission_final.csv', models, tokenizers, DEVICE, BATCH_SIZE)
print(f"Processed {total} samples")

## 3. Verify Output

In [ ]:
print("\n--- Verifying Output ---")

submission = pd.read_csv('submission_final.csv')
print(f"Submission rows: {len(submission)}")
print(submission.head())

sample = pd.read_csv('sample_submission.csv')
print(f"\nSample columns: {sample.columns.tolist()}")
print(f"Our columns:    {submission.columns.tolist()}")
print(f"Match: {list(sample.columns) == list(submission.columns)}")

## 4. Summary

In [ ]:
print("=" * 60)
print("SUBMISSION SUMMARY")
print("=" * 60)

models_used = list(models.keys())
print(f"""
Models used: {models_used}
Device used: {DEVICE}
Test samples: {total}
Output file: submission_final.csv
Batch size: {BATCH_SIZE}

Submission format verified against sample_submission.csv
""")

print("Files created:")
print("  - submission_final.csv: Final submission file")